# Cross-session Neural Trajectory Robustness

Run the finalized shape-only trajectory tests on the 20 ms activity reconstructed from an `s11` Willett representation export. The decoder remains fixed. All trajectory preprocessing, PCA, and class templates are fitted without the held-out analysis sessions.

Leave-one-session-out folds are the primary inferential units. Unique repeated 25% session splits are secondary stability diagnostics. Every real phoneme event is compared with five matched random centers from the same utterance.

In [1]:
# Colab / Drive / repository bootstrap.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Not running in Colab or Drive already unavailable: {exc}')

import os
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path('/content/utah-ssl') if Path('/content').exists() else Path.cwd()
REPO_URL = 'https://github.com/ethan-read/utah-ssl.git'
if str(REPO_DIR).startswith('/content'):
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', 'pull', '--ff-only'], cwd=str(REPO_DIR), check=False)
os.chdir(REPO_DIR)
PACKAGE_ROOT = REPO_DIR
os.environ['PYTHONPATH'] = f"{PACKAGE_ROOT}:{os.environ.get('PYTHONPATH', '')}"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

try:
    import pandas  # noqa: F401
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'pandas', 'matplotlib', 'scikit-learn'], check=True)
print('REPO_DIR:', REPO_DIR)

Mounted at /content/drive
REPO_DIR: /content/utah-ssl


## Configuration

The default target is the locally trained Brain2Text24 GRU export used in `s12`. The legacy export folder name is only a container name; `MODEL_KEY='gru'` selects the local model.

In [2]:
from experiments.manifolds import RobustnessConfig

DRIVE_ROOT = Path('/content/drive/MyDrive') if Path('/content/drive/MyDrive').exists() else Path('/Users/home/My Drive')
UTAH_SSL_ROOT = DRIVE_ROOT / 'utah_ssl'
REPRESENTATION_ROOT = UTAH_SSL_ROOT / 'data' / 'representations' / 'willett_manifolds'
OUTPUT_ROOT = UTAH_SSL_ROOT / 'outputs' / 'neural_trajectories'

EXPORT_NAME = 'stanford_released_gru_s5_val_area6v_soft_phonetic_categories_v1'
MODEL_KEY = 'gru'
MODEL_DIR = REPRESENTATION_ROOT / EXPORT_NAME / MODEL_KEY
REPRESENTATIONS = ('input_windows', 'adapted_input_windows')

CONFIG = RobustnessConfig(
    before_bins=15,
    after_bins=15,
    null_centers_per_event=5,
    null_exclusion_bins=30,  # Prevent any overlap with the +/-15-bin real path.
    min_train_events=20,
    min_test_events=4,
    max_train_events_per_phoneme=100,
    max_test_events_per_phoneme=100,
    primary_pca_components=6,
    sensitivity_pca_components=(24,),
    repeated_split_count=25,
    heldout_session_fraction=0.25,
    permutation_repetitions=200,
    bootstrap_repetitions=1000,
    seed=7,
)

print('MODEL_DIR:', MODEL_DIR)
if not (MODEL_DIR / 'metadata.json').exists():
    available = sorted(str(path.parent.relative_to(REPRESENTATION_ROOT)) for path in REPRESENTATION_ROOT.glob('*/*/metadata.json')) if REPRESENTATION_ROOT.exists() else []
    raise FileNotFoundError(f'Missing export: {MODEL_DIR}. Available model exports: {available}')

ImportError: cannot import name 'RobustnessConfig' from 'neural_trajectories' (/content/utah-ssl/experiments/manifolds/__init__.py)

## Run robustness analyses

This is the compute-heavy cell. For each representation it reconstructs the covered 20 ms bins once, runs every leave-one-session-out fold, evaluates 25 unique repeated session splits, and repeats LOSO with 24 PCs as a dimensionality sensitivity check.

In [ ]:
import json
import numpy as np
import pandas as pd
from experiments.manifolds import run_robustness_analysis, save_robustness_result

shards = json.loads((MODEL_DIR / 'shards.json').read_text())
with np.load(MODEL_DIR / 'shards' / shards[0]['shard']) as first_shard:
    available_arrays = set(first_shard.files)

robustness_results = {}
saved_outputs = {}
for representation in REPRESENTATIONS:
    if representation not in available_arrays:
        print(f'Skipping {representation}: not present in the export')
        continue
    print(f'\n=== {representation} ===')
    result = run_robustness_analysis(
        MODEL_DIR,
        representation=representation,
        config=CONFIG,
        progress=print,
    )
    output_dir = OUTPUT_ROOT / EXPORT_NAME / MODEL_KEY / f'{representation}_20ms_final_robustness'
    saved = save_robustness_result(result, output_dir)
    robustness_results[representation] = result
    saved_outputs[representation] = saved
    print(result.summary.to_string(index=False))
    print('Saved to:', output_dir)

## Primary results

The primary table summarizes paired real-minus-null effects across leave-one-session-out folds. Confidence intervals and sign tests use sessions—not individual phoneme events—as the sampling units.

In [ ]:
from IPython.display import Image, display

summary = pd.concat(
    [result.summary for result in robustness_results.values()],
    ignore_index=True,
)
display(summary)

for representation, result in robustness_results.items():
    print(f'\n{representation}: LOSO fold results')
    display(result.loso.sort_values('heldout_session').reset_index(drop=True))
    display(Image(filename=saved_outputs[representation]['loso_real_vs_null.png']))

## Stability, sensitivity, and diagnostics

Repeated splits are descriptive because their held-out session sets overlap. LOSO remains the inferential result. Inspect reconstruction diagnostics for overlap errors, missing tail bins, and excluded examples before interpreting small effects.

In [ ]:
for representation, result in robustness_results.items():
    print(f'\n=== {representation}: repeated 25% splits ===')
    display(result.repeated_splits.describe(include='all'))
    display(Image(filename=saved_outputs[representation]['repeated_split_real_vs_null.png']))

    print(f'{representation}: PCA sensitivity')
    sensitivity_summary = result.sensitivity.groupby(['components', 'confidence_subset']).agg(
        sessions=('heldout_session', 'count'),
        median_real_minus_null_separation=('real_minus_null_separation', 'median'),
        median_phoneme_accuracy=('real_phoneme_balanced_accuracy', 'median'),
        median_null_phoneme_accuracy=('null_phoneme_balanced_accuracy', 'median'),
        median_category_accuracy=('real_category_balanced_accuracy', 'median'),
        median_null_category_accuracy=('null_category_balanced_accuracy', 'median'),
    ).reset_index()
    display(sensitivity_summary)

    print(f'{representation}: reconstruction diagnostics')
    display(result.diagnostics.groupby('status').agg(
        examples=('example_index', 'count'),
        events=('event_count', 'sum'),
        boundary_excluded=('boundary_excluded_count', 'sum'),
        null_excluded=('null_excluded_count', 'sum'),
        maximum_overlap_error=('max_overlap_error', 'max'),
        unavailable_tail_bins=('unavailable_tail_bins', 'sum'),
    ).reset_index())

    if 'timecourses' in saved_outputs[representation]:
        display(Image(filename=saved_outputs[representation]['timecourses']))
    if 'distance_matrix' in saved_outputs[representation]:
        display(Image(filename=saved_outputs[representation]['distance_matrix']))

## Interpretation contract

Evidence for cross-session phoneme trajectory shape requires a positive LOSO real-minus-null effect whose session-bootstrap interval excludes zero, real balanced classification above both the matched-null and within-session label-permutation baselines, and broadly positive effects across individual sessions. Adapted-only evidence supports a learned common representation; a matching normalized-input result is closer to evidence in the recorded neural population activity. CTC timing remains model-assisted.

In [3]:
# Optional Colab resource release.
try:
    from google.colab import runtime
    runtime.unassign()
except Exception:
    pass